In [1]:
import pandas as pd
import numpy as np

In [2]:
orders = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\orders.csv")
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,7/4/2012,58578,1109,delivered,credit_card,desktop,paid_search
1,2,7/4/2012,58621,1330,returned,cod,mobile,paid_search
2,3,7/4/2012,58811,1473,delivered,credit_card,desktop,direct
3,4,7/4/2012,59453,2360,delivered,credit_card,desktop,referral
4,20,7/4/2012,42962,8620,delivered,credit_card,tablet,organic_search


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   order_id        646945 non-null  int64
 1   order_date      646945 non-null  str  
 2   customer_id     646945 non-null  int64
 3   zip             646945 non-null  int64
 4   order_status    646945 non-null  str  
 5   payment_method  646945 non-null  str  
 6   device_type     646945 non-null  str  
 7   order_source    646945 non-null  str  
dtypes: int64(3), str(5)
memory usage: 39.5 MB


In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders.dtypes

order_id                   int64
order_date        datetime64[us]
customer_id                int64
zip                        int64
order_status                 str
payment_method               str
device_type                  str
order_source                 str
dtype: object

In [5]:
orders.isnull().sum()

order_id          0
order_date        0
customer_id       0
zip               0
order_status      0
payment_method    0
device_type       0
order_source      0
dtype: int64

In [6]:
orders[orders.duplicated()]

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source


In [7]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,20,2012-07-04,42962,8620,delivered,credit_card,tablet,organic_search


## Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [9]:
counts = orders['customer_id'].value_counts().reset_index()
counts[counts['count'] == 1]['customer_id'].reset_index()

,index,customer_id
0,67888,3479
1,67889,39584
2,67890,53240
3,67891,30676
4,67892,123891
...,...,...
22353,90241,89041
22354,90242,123684
22355,90243,123687
22356,90244,94649


In [10]:
filtered_orders = orders.groupby('customer_id').filter(lambda x: len(x) > 1)

In [11]:
filtered_orders

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,20,2012-07-04,42962,8620,delivered,credit_card,tablet,organic_search
...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct


In [12]:
filtered_orders = filtered_orders.sort_values(['customer_id', 'order_date'])
filtered_orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
3737,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search
143952,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral
239745,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search
374586,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search
544495,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search


In [13]:
filtered_orders['prev_date'] = filtered_orders.groupby('customer_id')['order_date'].shift(1)
filtered_orders['gap_days'] = (filtered_orders['order_date'] - filtered_orders['prev_date']).dt.days
filtered_orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,prev_date,gap_days
3737,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,NaT,NaN
143952,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,2012-07-25,675.0
239745,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,2014-05-31,426.0
374586,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,2015-07-31,632.0
544495,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,2017-04-23,1037.0


In [14]:
gaps = filtered_orders.dropna(subset=['gap_days'])
gaps.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,prev_date,gap_days
143952,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,2012-07-25,675.0
239745,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,2014-05-31,426.0
374586,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,2015-07-31,632.0
544495,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,2017-04-23,1037.0
586956,756884,2021-04-24,1,15201,paid,credit_card,desktop,social_media,2020-02-24,425.0


In [15]:
median_gap = gaps['gap_days'].median()
median_gap

144.0

## Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [17]:
products = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\products.csv")
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633160,11371.919280
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717300,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334540,14063.570410


In [18]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 2412 entries, 0 to 2411
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    2412 non-null   int64  
 1   product_name  2412 non-null   str    
 2   category      2412 non-null   str    
 3   segment       2412 non-null   str    
 4   size          2412 non-null   str    
 5   color         2412 non-null   str    
 6   price         2412 non-null   float64
 7   cogs          2412 non-null   float64
dtypes: float64(2), int64(1), str(5)
memory usage: 150.9 KB


In [19]:
products['segment'].unique()

<StringArray>
[   'Everyday', 'Performance',    'Balanced',    'Standard', 'All-weather',
     'Premium',      'Trendy',  'Activewear']
Length: 8, dtype: str

In [20]:
products['gross_margin'] = (products['price']-products['cogs'])/products['price']
products.head()

,product_id,product_name,category,segment,size,color,price,cogs,gross_margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633160,11371.919280,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717300,8573.172954,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334540,14063.570410,0.1080


In [21]:
avg_gross_margin = products.groupby('segment')['gross_margin'].mean().reset_index()
avg_gross_margin.sort_values(by='gross_margin', ascending=False)

,segment,gross_margin
6,Standard,0.313442
5,Premium,0.285377
1,All-weather,0.284176
0,Activewear,0.265600
4,Performance,0.263650
2,Balanced,0.258038
7,Trendy,0.240758
3,Everyday,0.236343


## Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [23]:
returns = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\returns.csv")
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [24]:
returns.info()

<class 'pandas.DataFrame'>
RangeIndex: 39939 entries, 0 to 39938
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   return_id        39939 non-null  str    
 1   order_id         39939 non-null  int64  
 2   product_id       39939 non-null  int64  
 3   return_date      39939 non-null  str    
 4   return_reason    39939 non-null  str    
 5   return_quantity  39939 non-null  int64  
 6   refund_amount    39939 non-null  float64
dtypes: float64(1), int64(3), str(3)
memory usage: 2.1 MB


In [25]:
returns_merged = returns.merge(
    products[['product_id', 'category']],
    on='product_id',
    how='left'
)
returns_merged

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,category
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,Streetwear
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,GenZ
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,Streetwear
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,Outdoor
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,Outdoor
...,...,...,...,...,...,...,...,...
39934,RET-051470,832867,653,2022-12-27,wrong_size,3,24741.62,Streetwear
39935,RET-051471,832890,792,2022-12-30,late_delivery,1,560.50,Outdoor
39936,RET-051481,833005,449,2022-12-31,defective,1,10002.55,Streetwear
39937,RET-051494,833234,1085,2022-12-28,wrong_size,1,815.57,Outdoor


In [26]:
filtered_returns = returns_merged.loc[returns_merged['category'] == 'Streetwear']
filtered_returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,category
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,Streetwear
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,Streetwear
5,RET-000006,59,671,2012-07-19,defective,1,10086.33,Streetwear
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22,Streetwear
7,RET-000008,102,467,2012-07-17,defective,1,9724.09,Streetwear


In [27]:
filtered_returns['return_reason'].value_counts()

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

## Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [29]:
traffic = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\web_traffic.csv")
traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,1/1/2013,9760,7253,39093,0.00514,102.9,organic_search
1,1/2/2013,10456,8151,47611,0.00406,120.5,organic_search
2,1/3/2013,10076,7458,36963,0.00401,263.6,direct
3,1/4/2013,9973,8063,53078,0.00562,151.8,direct
4,1/5/2013,10223,7882,36790,0.00525,168.6,referral


In [30]:
traffic.info()

<class 'pandas.DataFrame'>
RangeIndex: 3652 entries, 0 to 3651
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3652 non-null   str    
 1   sessions                  3652 non-null   int64  
 2   unique_visitors           3652 non-null   int64  
 3   page_views                3652 non-null   int64  
 4   bounce_rate               3652 non-null   float64
 5   avg_session_duration_sec  3652 non-null   float64
 6   traffic_source            3652 non-null   str    
dtypes: float64(2), int64(3), str(2)
memory usage: 199.8 KB


In [31]:
traffic.groupby('traffic_source')['bounce_rate'].mean().reset_index().sort_values(by='bounce_rate', ascending=True)

,traffic_source,bounce_rate
1,email_campaign,0.004458
5,social_media,0.004476
3,paid_search,0.004478
4,referral,0.004499
2,organic_search,0.004504
0,direct,0.004511


## Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [33]:
order_items = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\order_items.csv")
order_items.head()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_16680\320677693.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [34]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  str    
 6   promo_id_2       206 non-null     str    
dtypes: float64(2), int64(3), str(2)
memory usage: 38.2 MB


In [35]:
notnull = order_items['promo_id'].notna().sum()
notnull

276316

In [36]:
null = order_items['promo_id'].isnull().sum()
null

438353

In [37]:
applied_promo_orders = round((notnull/(notnull+null))*100,2)
applied_promo_orders

38.66

## Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [71]:
customers = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\customers.csv")
customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,12/30/2021,Female,35-44,social_media
1,2,15201,Hai Phong,12/27/2013,Female,45-54,email_campaign
2,3,15201,Hai Phong,7/24/2018,Female,18-24,organic_search
3,4,15201,Hai Phong,11/29/2017,Male,35-44,referral
4,5,15201,Hai Phong,9/23/2022,Male,55+,organic_search


In [87]:
age_group_count = customers['age_group'].value_counts().reset_index()
age_group_count

,age_group,count
0,25-34,36342
1,35-44,31920
2,45-54,23172
3,18-24,17039
4,55+,13457


In [79]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,20,2012-07-04,42962,8620,delivered,credit_card,tablet,organic_search


In [81]:
orders_merged = orders.merge(
    customers[['customer_id', 'age_group']],
    on='customer_id',
    how='left'
)
orders_merged.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,age_group
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,35-44
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,18-24
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,35-44
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,45-54
4,20,2012-07-04,42962,8620,delivered,credit_card,tablet,organic_search,45-54


In [101]:
orders_age_group = orders_merged.groupby('age_group')['order_id'].count().reset_index()
final_age_group = age_group_count.merge(
    orders_age_group,
    on='age_group',
    how='left'
)
final_age_group = final_age_group.rename(columns={
    'count': 'number_customers',
    'order_id': 'number_orders'
})

In [107]:
final_age_group['avg_orders/customers'] = final_age_group['number_orders']/final_age_group['number_customers']
final_age_group.sort_values(by='avg_orders/customers',ascending=False)

,age_group,number_customers,number_orders,avg_orders/customers
4,55+,13457,72760,5.406851
2,45-54,23172,124138,5.357241
1,35-44,31920,170368,5.337343
0,25-34,36342,190622,5.245226
3,18-24,17039,89057,5.226656


## Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [110]:
sales_train = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\sales.csv")
sales_train

,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79
...,...,...,...
3828,2022-12-27,2100553.66,2184872.24
3829,2022-12-28,3448729.20,3513621.00
3830,2022-12-29,3083944.33,3170787.10
3831,2022-12-30,2884668.76,3022292.15


In [112]:
geography = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\geography.csv")
geography

,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13
...,...,...,...,...
39943,59933,Pleiku,West,District #33
39944,59934,Soc Trang,West,District #33
39945,59935,Rach Gia,West,District #33
39946,59936,Vung Tau,West,District #33


In [118]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,20,2012-07-04,42962,8620,delivered,credit_card,tablet,organic_search


In [120]:
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [122]:
order_items['revenue'] = order_items['quantity']*order_items['unit_price']
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,revenue
0,1,2400,7,1138.22,0.0,NaN,NaN,7967.54
1,2,609,7,10166.25,0.0,NaN,NaN,71163.75
2,3,396,3,11220.33,0.0,NaN,NaN,33660.99
3,4,635,5,10639.25,0.0,NaN,NaN,53196.25
4,6,1935,1,1597.84,0.0,NaN,NaN,1597.84


In [126]:
merged_orders_zip = order_items.merge(orders[['order_id','zip']], on='order_id',how='left')
final_merge = merged_orders_zip.merge(geography[['zip','region']], on='zip',how='left')
final_merge

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,revenue,zip,region
0,1,2400,7,1138.22,0.0,NaN,NaN,7967.54,1109,East
1,2,609,7,10166.25,0.0,NaN,NaN,71163.75,1330,East
2,3,396,3,11220.33,0.0,NaN,NaN,33660.99,1473,East
3,4,635,5,10639.25,0.0,NaN,NaN,53196.25,2360,East
4,6,1935,1,1597.84,0.0,NaN,NaN,1597.84,2886,East
...,...,...,...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN,35791.36,33907,East
714665,834377,1995,7,5250.79,0.0,NaN,NaN,36755.53,37091,East
714666,834387,2331,8,7389.06,0.0,NaN,NaN,59112.48,80516,Central
714667,834392,1115,5,4767.33,0.0,NaN,NaN,23836.65,93510,West


In [128]:
final_merge.groupby('region')['revenue'].sum().reset_index()

,region,revenue
0,Central,4.941908e+09
1,East,7.637533e+09
2,West,3.851035e+09


## Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [137]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
cancelled_orders['payment_method'].value_counts().sort_values(ascending=False)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

## Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [142]:
products.head()

,product_id,product_name,category,segment,size,color,price,cogs,gross_margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633160,11371.919280,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717300,8573.172954,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334540,14063.570410,0.1080


In [144]:
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [148]:
size_merged_returns = returns.merge(products[['product_id','size']],on=['product_id'],how='left')
size_merged_returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,size
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,M
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,L
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,XL
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,M
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,L


In [154]:
size_returns = size_merged_returns.groupby('size')['return_id'].count().reset_index()
size_returns

,size,return_id
0,L,9741
1,M,9820
2,S,9723
3,XL,10655


In [146]:
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,revenue
0,1,2400,7,1138.22,0.0,NaN,NaN,7967.54
1,2,609,7,10166.25,0.0,NaN,NaN,71163.75
2,3,396,3,11220.33,0.0,NaN,NaN,33660.99
3,4,635,5,10639.25,0.0,NaN,NaN,53196.25
4,6,1935,1,1597.84,0.0,NaN,NaN,1597.84


In [150]:
size_merged_order_items = order_items.merge(products[['product_id','size']],on=['product_id'],how='left')
size_merged_order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,revenue,size
0,1,2400,7,1138.22,0.0,NaN,NaN,7967.54,S
1,2,609,7,10166.25,0.0,NaN,NaN,71163.75,M
2,3,396,3,11220.33,0.0,NaN,NaN,33660.99,S
3,4,635,5,10639.25,0.0,NaN,NaN,53196.25,XL
4,6,1935,1,1597.84,0.0,NaN,NaN,1597.84,XL


In [166]:
size_orders = size_merged_order_items.groupby('size')['order_id'].count().reset_index()
size_orders_vs_returns = size_orders.merge(size_returns,on='size',how='left')

In [174]:
size_orders_vs_returns = size_orders_vs_returns.rename(columns={
    'return_id': 'number_returns',
    'order_id': 'number_orders'
})
size_orders_vs_returns['return_rate'] = size_orders_vs_returns['number_returns'] / size_orders_vs_returns['number_orders']
size_orders_vs_returns.sort_values(by='return_rate',ascending=False)

,size,number_orders,number_returns,return_rate
2,S,172042,9723,0.056515
0,L,173174,9741,0.056250
1,M,176428,9820,0.055660
3,XL,193025,10655,0.055200


## Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?

In [179]:
payments = pd.read_csv("D:\\VinchallengeDatathon\\datathon-2026-round-1\\payments.csv")
payments.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [181]:
payments['payment_method'].value_counts()

payment_method
credit_card      356352
paypal            97018
cod               96681
apple_pay         64763
bank_transfer     32131
Name: count, dtype: int64

In [185]:
payments_by_periods = payments.groupby('installments')['payment_value'].mean().reset_index()
payments_by_periods.sort_values(by='payment_value',ascending=False)

,installments,payment_value
3,6,24446.654403
2,3,24399.635486
4,12,24245.772694
0,1,24113.274166
1,2,708.473729
